# ETF V6 anchor + ML filter + dynamic de-dup + governance experiment

目的：在固定干净 ETF 池上，比较手动趋势 anchor、LGB 模型、动态相似度去重、手动 top1 + ML filter/scale、以及简单 model governance。

核心假设：
- 手动趋势/R2 是强 anchor，不应轻易替代。
- ML 的价值可能在直接选 ETF，也可能在过滤手动信号。
- top3 组合如果同质化严重，需要用收益路径相关性动态去重，而不是写死行业限制。
- 最终要输出可筛选的候选策略，而不是只追求单条曲线。

输出：`etf_ml_v6_*` 系列 CSV。

In [ ]:
# =========================
# 0. Config
# =========================
from jqdata import *
import os
import gc
import math
import pickle
import datetime
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OUT_DIR = "etf_ml_v6_anchor_ml_filter_dedup_governance_outputs"
USE_ORTHOGONAL_POOL = True
ORTHOGONAL_POOL_PATH = "etf_v4_fixed_orthogonal_pool_outputs/etf_orthogonal_pool_v1.csv"
PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v6_weekly_panel.csv")
PATH_PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v6_path_panel.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v6_score_panel.csv")
WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v6_weekly_portfolio_proxy.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v6_summary.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v6_latest_targets.csv")
MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v6_model_manifest.csv")
MODEL_BUNDLE_DIR = os.path.join(OUT_DIR, "models")
STRATEGY_COMPARE_CSV = os.path.join(OUT_DIR, "etf_ml_v6_strategy_compare.csv")
DEDUP_REPORT_CSV = os.path.join(OUT_DIR, "etf_ml_v6_dedup_report.csv")
FILTER_REPORT_CSV = os.path.join(OUT_DIR, "etf_ml_v6_filter_report.csv")
GOVERNANCE_REPORT_CSV = os.path.join(OUT_DIR, "etf_ml_v6_governance_report.csv")

# Default: reuse cached panel if it exists. Set FORCE_REBUILD_DATA=True to rebuild from JoinQuant API.
FORCE_REBUILD_DATA = False
REBUILD_DATA = bool(FORCE_REBUILD_DATA or (not os.path.exists(PANEL_CSV)))
EXPORT_MODELS = True
START_DATE = "2019-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
PATH_LOOKBACK_DAYS = 120
HORIZON_DAYS_LIST = [3, 5, 10]
LOWMEAN_DAYS_LIST = [10, 20]
MAX_HORIZON_DAYS = max(HORIZON_DAYS_LIST)
MAX_LABEL_DAYS = max(HORIZON_DAYS_LIST + LOWMEAN_DAYS_LIST)
MIN_LISTING_DAYS = 180
MIN_AVG_MONEY_20 = 20000000.0
ETF_CHUNK_SIZE = 120
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42

SELECT_MODES = ["top1", "top3", "corr_dedup_top3", "kmeans_top3"]
DEDUP_CANDIDATE_N = 10
DEDUP_TARGET_N = 3
DEDUP_CORR_THRESHOLD = 0.90
DEDUP_MIN_PATH_OBS = 40

FILTER_BUY_RANK_PCT = 0.60
SCALE_LOW_RANK_PCT = 0.40
SCALE_HIGH_RANK_PCT = 0.70
GOVERNANCE_LOOKBACK_LIST = [12, 24]
GOVERNANCE_MIN_OBS = 8
GOVERNANCE_MAX_DD_FLOOR = -0.35

TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1, PATH_LOOKBACK_DAYS + 1)
POOL_CONTEXT_WINDOW = 25
RISK_BREADTH_COL = "pool_breadth_25"
CASH_RETURN = 0.0

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
    "短融", "中票", "AAA", "信用", "0-3", "1-3", "政策性金融债",
]
ETF_NAME_CACHE = {}

TRAIN_WINDOWS = [
    ("train20210101_20231231", "2021-01-01", "2023-12-31"),
    ("train20190101_20241231", "2019-01-01", "2024-12-31"),
    ("train20210101_20251231", "2021-01-01", "2025-12-31"),
]

TARGET_SPECS = [
    {"target_name": "ret3d", "target_col": "future_ret_3d", "eval_ret_col": "future_ret_3d", "next_col": "next_date_3d", "eval_horizon_days": 3},
    {"target_name": "ret5d", "target_col": "future_ret_5d", "eval_ret_col": "future_ret_5d", "next_col": "next_date_5d", "eval_horizon_days": 5},
    {"target_name": "ret10d", "target_col": "future_ret_10d", "eval_ret_col": "future_ret_10d", "next_col": "next_date_10d", "eval_horizon_days": 10},
    {"target_name": "lowmean10d", "target_col": "future_low_mean_ret_10d", "eval_ret_col": "future_ret_10d", "next_col": "next_date_10d", "eval_horizon_days": 10},
    {"target_name": "lowmean20d", "target_col": "future_low_mean_ret_20d", "eval_ret_col": "future_ret_10d", "next_col": "next_date_10d", "eval_horizon_days": 10},
]
MANUAL_TARGET_NAME = "manual_weekly"
MANUAL_EVAL_RET_COL = "future_ret_5d"
MANUAL_NEXT_COL = "next_date_5d"
MANUAL_HORIZON_DAYS = 5

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60",
    "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60",
    "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for _w in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % _w,
        "trend_r2_%s" % _w,
        "trend_score_%s" % _w,
        "trend_vol_%s" % _w,
        "trend_score_vol_adj_%s" % _w,
        "trend_simple_ann_%s" % _w,
    ])
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + c for c in RAW_FEATURE_COLS]
POOL_FEATURE_COLS = ["pool_breadth_%s" % POOL_CONTEXT_WINDOW, "pool_median_vol_%s" % POOL_CONTEXT_WINDOW]
FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + POOL_FEATURE_COLS
PATH_RET_COLS = ["path_ret_%03d" % i for i in range(1, PATH_LOOKBACK_DAYS + 1)]
LABEL_COLS = []
for _h in HORIZON_DAYS_LIST:
    LABEL_COLS.append("future_ret_%sd" % _h)
    LABEL_COLS.append("next_date_%sd" % _h)
for _h in LOWMEAN_DAYS_LIST:
    LABEL_COLS.append("future_low_mean_ret_%sd" % _h)
META_COLS = ["code", "name", "feature_date", "rebalance_date"] + LABEL_COLS

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "verbose": -1,
    "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 160

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
print("V6 output dir:", OUT_DIR)
print("panel cache exists:", os.path.exists(PANEL_CSV), "force rebuild:", FORCE_REBUILD_DATA, "REBUILD_DATA:", REBUILD_DATA)
print("feature count:", len(FEATURE_COLS), "path cols:", len(PATH_RET_COLS), "target specs:", [x["target_name"] for x in TARGET_SPECS])



In [ ]:
# =========================
# 0b. Fixed current orthogonal ETF pool
# =========================
def load_orthogonal_pool(path):
    if not USE_ORTHOGONAL_POOL:
        return pd.DataFrame(), set()
    if not os.path.exists(path):
        raise IOError("orthogonal pool file not found: " + path + "\nRun ETF_V4_fixed_current_orthogonal_pool_builder实验.ipynb first. V5 requires the fixed orthogonal pool.")
    pool = pd.read_csv(path)
    if "code" not in pool.columns:
        raise ValueError("orthogonal pool csv missing code column: " + path)
    pool["code"] = pool["code"].astype(str)
    codes = set(pool["code"].dropna().astype(str).tolist())
    if len(codes) == 0:
        raise ValueError("orthogonal pool is empty: " + path)
    return pool, codes


def filter_to_orthogonal_pool(df):
    if (not USE_ORTHOGONAL_POOL) or df is None or df.empty:
        return df
    if "code" not in df.columns:
        return df
    before = df.shape[0]
    out = df[df["code"].astype(str).isin(ORTHOGONAL_POOL_CODES)].copy()
    print("orthogonal pool filter rows:", before, "->", out.shape[0])
    return out


orthogonal_pool_df, ORTHOGONAL_POOL_CODES = load_orthogonal_pool(ORTHOGONAL_POOL_PATH)
print("orthogonal pool size:", len(ORTHOGONAL_POOL_CODES))
if USE_ORTHOGONAL_POOL:
    display(orthogonal_pool_df.head(30))

In [ ]:
# =========================
# 1. Data helpers: JoinQuant weekly ETF panel
# =========================
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_weekly_feature_dates(start_date, end_date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("数据重建需要在 JoinQuant 研究环境运行；如果已有缓存，请设置 REBUILD_DATA=False。")
    if len(days) == 0:
        return []
    s = pd.Series(days)
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%W")):
        out.append(pd.Timestamp(gdf.max()).normalize())
    return out


def should_exclude_name(name):
    text = str(name)
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and kw in text:
            return True
    return False


def get_etf_universe_on_date(date):
    try:
        sec_df = get_all_securities(["etf"], date=date)
    except NameError:
        raise RuntimeError("get_all_securities 不可用：请在 JoinQuant 研究环境运行数据重建。")
    except Exception as err:
        print("get_all_securities failed", date, err)
        return []
    if sec_df is None or sec_df.empty:
        return []
    out = []
    name_map = {}
    for code, row in sec_df.iterrows():
        try:
            start_date = row.get("start_date", None)
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
            feature_dt = pd.Timestamp(date).date()
            if feature_dt - start_date < datetime.timedelta(days=MIN_LISTING_DAYS):
                continue
            name = row.get("display_name", "")
            if should_exclude_name(name):
                continue
            out.append(code)
            name_map[code] = str(name)
        except Exception:
            continue
    if USE_ORTHOGONAL_POOL:
        out = [code for code in out if str(code) in ORTHOGONAL_POOL_CODES]
        name_map = dict((code, name_map.get(code, "")) for code in out)
    ETF_NAME_CACHE[str(pd.Timestamp(date).date())] = name_map
    return out


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def calc_trend_metrics(close, days):
    empty = {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    if len(close) <= days:
        return empty
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return empty
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    try:
        slope, intercept = np.polyfit(x, y, 1, w=weights)
    except Exception:
        return empty
    ann = math.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * math.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {"ann": ann, "r2": r2, "score": score, "vol": vol, "score_vol_adj": score_vol_adj, "simple_ann": simple_ann}


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    for col in ["open", "high", "low", "close", "volume", "money"]:
        if col not in df.columns:
            return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None

    ret = close.pct_change()
    rec = {"code": code}
    path_values = ret.tail(PATH_LOOKBACK_DAYS).values
    if len(path_values) < PATH_LOOKBACK_DAYS:
        pad = [np.nan] * (PATH_LOOKBACK_DAYS - len(path_values))
        path_values = np.array(pad + list(path_values))
    for _idx, _val in enumerate(path_values, 1):
        rec["path_ret_%03d" % _idx] = _val
    rec["ret_1"] = safe_ret(close, 1)
    rec["ret_5"] = safe_ret(close, 5)
    rec["ret_10"] = safe_ret(close, 10)
    rec["ret_20"] = safe_ret(close, 20)
    rec["ret_60"] = safe_ret(close, 60)
    rec["vol_5"] = ret.tail(5).std()
    rec["vol_20"] = ret.tail(20).std()
    rec["vol_60"] = ret.tail(60).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1.0
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1.0
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1.0
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1.0
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1.0
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1.0
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1.0
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1.0
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1.0
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1.0
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for w in TREND_WINDOWS:
        tm = calc_trend_metrics(close, w)
        rec["trend_ann_%s" % w] = tm["ann"]
        rec["trend_r2_%s" % w] = tm["r2"]
        rec["trend_score_%s" % w] = tm["score"]
        rec["trend_vol_%s" % w] = tm["vol"]
        rec["trend_score_vol_adj_%s" % w] = tm["score_vol_adj"]
        rec["trend_simple_ann_%s" % w] = tm["simple_ann"]
    return rec


def fetch_feature_rows(feature_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            price_df = get_price(
                etf_chunk,
                end_date=feature_date,
                frequency="daily",
                fields=fields,
                count=PRICE_HISTORY_COUNT,
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("price feature chunk failed", feature_date, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    if len(rows) == 0:
        return pd.DataFrame()
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)


def fetch_future_returns_multi(feature_date, etfs):
    ret_cols = ["future_ret_%sd" % h for h in HORIZON_DAYS_LIST]
    next_cols = ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]
    low_cols = ["future_low_mean_ret_%sd" % h for h in LOWMEAN_DAYS_LIST]
    try:
        td = pd.to_datetime(get_trade_days(start_date=feature_date, count=MAX_LABEL_DAYS + 1))
    except Exception as err:
        print("future trade days failed", feature_date, err)
        return pd.DataFrame(columns=["code"] + ret_cols + next_cols + low_cols)
    if len(td) < MAX_LABEL_DAYS + 1:
        return pd.DataFrame(columns=["code"] + ret_cols + next_cols + low_cols)
    next_date_map = {}
    for h in HORIZON_DAYS_LIST:
        next_date_map[h] = pd.Timestamp(td[h]).normalize()
    max_next_date = pd.Timestamp(td[MAX_LABEL_DAYS]).normalize()

    rows = []
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            px = get_price(
                etf_chunk,
                start_date=feature_date,
                end_date=max_next_date,
                frequency="daily",
                fields=["close", "low"],
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("future price chunk failed", feature_date, err)
            continue
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time").copy()
            if len(one) < 2:
                continue
            one["time_norm"] = pd.to_datetime(one["time"]).dt.normalize()
            close_by_date = dict(zip(one["time_norm"], one["close"].astype(float)))
            start_dt = pd.Timestamp(feature_date).normalize()
            if start_dt not in close_by_date or close_by_date[start_dt] <= 0:
                continue
            start_close = float(close_by_date[start_dt])
            rec = {"code": code}
            ok = False
            for h in HORIZON_DAYS_LIST:
                nd = next_date_map[h]
                rec["next_date_%sd" % h] = nd
                if nd in close_by_date:
                    rec["future_ret_%sd" % h] = float(close_by_date[nd]) / start_close - 1.0
                    ok = True
                else:
                    rec["future_ret_%sd" % h] = np.nan
            future_one = one[one["time_norm"] > start_dt].copy()
            for h in LOWMEAN_DAYS_LIST:
                lows = future_one.head(h)["low"].astype(float).dropna() if "low" in future_one.columns else pd.Series([])
                if len(lows) >= max(1, int(h * 0.6)):
                    rec["future_low_mean_ret_%sd" % h] = float(lows.mean()) / start_close - 1.0
                    ok = True
                else:
                    rec["future_low_mean_ret_%sd" % h] = np.nan
            if ok:
                rows.append(rec)
    return pd.DataFrame(rows)


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        if col in out.columns:
            out["rank_" + col] = out[col].rank(pct=True)
    return out


def add_pool_context_features(df):
    out = df.copy()
    ann_col = "trend_ann_%s" % POOL_CONTEXT_WINDOW
    vol_col = "trend_vol_%s" % POOL_CONTEXT_WINDOW
    if ann_col in out.columns and len(out) > 0:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = float((out[ann_col] > 0).mean())
    else:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = np.nan
    if vol_col in out.columns and len(out) > 0:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = float(out[vol_col].median())
    else:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = np.nan
    return out


def build_one_feature_date(feature_date):
    etfs = get_etf_universe_on_date(feature_date)
    if len(etfs) == 0:
        return pd.DataFrame()
    feature_df = fetch_feature_rows(feature_date, etfs)
    if feature_df.empty:
        return pd.DataFrame()
    if "money_mean_20" in feature_df.columns:
        feature_df = feature_df[feature_df["money_mean_20"] >= MIN_AVG_MONEY_20].copy()
    if feature_df.empty:
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    label_df = fetch_future_returns_multi(feature_date, list(feature_df["code"]))
    if label_df.empty:
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    if out.empty:
        return out
    name_map = ETF_NAME_CACHE.get(str(pd.Timestamp(feature_date).date()), {})
    out["name"] = out["code"].map(name_map).fillna("")
    out = add_rank_features(out)
    out["feature_date"] = pd.Timestamp(feature_date).normalize()
    out["rebalance_date"] = out["feature_date"]
    return out

In [ ]:
# =========================
# 2. Build or load weekly ETF panel
# =========================
def drop_incomplete_label_dates(panel):
    out = panel.copy()
    bad_dates = set()
    check_cols = ["future_ret_%sd" % h for h in HORIZON_DAYS_LIST]
    for h in LOWMEAN_DAYS_LIST:
        check_cols.append("future_low_mean_ret_%sd" % h)
    for col in check_cols:
        if col not in out.columns:
            continue
        q = out.groupby("feature_date")[col].agg(["count", lambda s: (s == 0).mean()])
        q.columns = ["sample_count", "zero_return_rate"]
        for dt in q[q["zero_return_rate"] >= 0.98].index:
            bad_dates.add(dt)
    if bad_dates:
        print("drop incomplete label dates:", sorted([str(pd.Timestamp(x).date()) for x in bad_dates]))
        out = out[~out["feature_date"].isin(bad_dates)].copy()
    return out


def find_excluded_name_rows(df, name_col):
    if df is None or df.empty or name_col not in df.columns:
        return pd.DataFrame()
    mask = pd.Series(False, index=df.index)
    names = df[name_col].astype(str)
    for kw in EXCLUDE_NAME_KEYWORDS:
        mask = mask | names.str.contains(kw, na=False)
    return df.loc[mask].copy()


def assert_no_excluded_names(df, name_col, label):
    bad = find_excluded_name_rows(df, name_col)
    if bad.empty:
        print(label + " excluded-name check: OK")
        return
    cols = []
    for c in ["feature_date", "code", "name", "targets", "target_names", "model_name", "select_mode", "risk_mode"]:
        if c in bad.columns and c not in cols:
            cols.append(c)
    print(label + " excluded-name check failed, rows=", len(bad))
    if cols:
        print(bad[cols].head(20).to_string(index=False))
    raise ValueError(label + " contains excluded ETF names. Rebuild data or clean old cache before trusting outputs.")


def build_weekly_panel():
    feature_dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("feature weeks:", len(feature_dates), "from", feature_dates[0] if feature_dates else None, "to", feature_dates[-1] if feature_dates else None)
    parts = []
    for dt in tqdm(feature_dates, desc="build etf weekly panel"):
        one = build_one_feature_date(dt)
        if one is not None and not one.empty:
            parts.append(one)
        if len(parts) % 20 == 0:
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("No ETF weekly samples were built. Check ETF universe/data access.")
    panel = pd.concat(parts, ignore_index=True)
    for c in ["feature_date", "rebalance_date"]:
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    for h in HORIZON_DAYS_LIST:
        c = "next_date_%sd" % h
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    return drop_incomplete_label_dates(panel)


def required_panel_label_cols():
    cols = []
    for h in HORIZON_DAYS_LIST:
        cols.append("future_ret_%sd" % h)
        cols.append("next_date_%sd" % h)
    for h in LOWMEAN_DAYS_LIST:
        cols.append("future_low_mean_ret_%sd" % h)
    return cols


def load_or_build_weekly_panel():
    need_build = bool(REBUILD_DATA or (not os.path.exists(PANEL_CSV)))
    panel = None
    if not need_build:
        panel = pd.read_csv(PANEL_CSV)
        missing_cols = [c for c in required_panel_label_cols() if c not in panel.columns]
        if missing_cols:
            print("cached panel missing label columns, rebuild required:", missing_cols)
            need_build = True
    if need_build:
        panel = build_weekly_panel()
        panel.to_csv(PANEL_CSV, index=False)
    date_cols = ["feature_date", "rebalance_date"] + ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]
    for c in date_cols:
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    return panel


panel_df = load_or_build_weekly_panel()
panel_df = filter_to_orthogonal_pool(panel_df)
assert_no_excluded_names(panel_df, "name", "panel_df")
print("panel shape:", panel_df.shape)
print(panel_df[["feature_date"] + ["next_date_%sd" % h for h in HORIZON_DAYS_LIST]].agg(["min", "max"]))
print("sample per week:")
print(panel_df.groupby("feature_date")["code"].count().describe())
print("missing labels:")
print(panel_df[["future_ret_%sd" % h for h in HORIZON_DAYS_LIST]].isnull().mean())
display(panel_df.head())


In [ ]:
# =========================
# 3. Scoring models: LGB targets and manual trend anchor
# =========================
def clean_feature_frame(df, include_path=False):
    cols = []
    source_cols = META_COLS + FEATURE_COLS
    if include_path:
        source_cols = source_cols + PATH_RET_COLS
    for c in source_cols:
        if c in df.columns and c not in cols:
            cols.append(c)
    return df.loc[:, cols].replace([np.inf, -np.inf], np.nan).copy()


def get_target_spec(target_name):
    for spec in TARGET_SPECS:
        if spec["target_name"] == target_name:
            return spec
    return None


def manual_target_for_eval_horizon(eval_horizon_days):
    # Manual trend/R2 is one fixed formula. The horizon is only used by ML labels,
    # so all manual comparisons should point to the same weekly anchor.
    return MANUAL_TARGET_NAME


def train_lgb_or_fallback(train_df, target_col):
    train_df = train_df[~train_df[target_col].isnull()].copy()
    X_raw = train_df.reindex(columns=FEATURE_COLS).replace([np.inf, -np.inf], np.nan)
    fill_values = X_raw.median().to_dict()
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    y = train_df[target_col].astype(float).values
    try:
        import lightgbm as lgb
        dtrain = lgb.Dataset(X[FEATURE_COLS], label=y, feature_name=list(FEATURE_COLS), free_raw_data=True)
        model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
        backend = "lightgbm"
        del dtrain
    except Exception as err:
        print("LightGBM unavailable or failed, fallback to sklearn RandomForestRegressor:", err)
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(
            n_estimators=240,
            max_depth=6,
            min_samples_leaf=25,
            random_state=RANDOM_SEED,
            n_jobs=1,
        )
        model.fit(X[FEATURE_COLS], y)
        backend = "sklearn_random_forest"
    del X_raw, X, y
    gc.collect()
    return model, fill_values, backend


def predict_model(model, fill_values, df):
    X_raw = df.reindex(columns=FEATURE_COLS).replace([np.inf, -np.inf], np.nan)
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    pred = np.asarray(model.predict(X[FEATURE_COLS])).reshape(-1).astype(float)
    del X_raw, X
    gc.collect()
    return pred


def calc_manual_trend_score(df):
    out = pd.Series(0.0, index=df.index)
    weights = [
        ("rank_trend_score_vol_adj_25", 0.40),
        ("rank_trend_score_20", 0.20),
        ("rank_trend_r2_25", 0.15),
        ("rank_ret_20", 0.10),
        ("rank_drawdown_20", 0.10),
        ("rank_money_ratio_5_20", 0.05),
        ("rank_vol_20", -0.10),
    ]
    total_abs = 0.0
    for col, w in weights:
        if col in df.columns:
            out = out + df[col].fillna(df[col].median()).fillna(0.5) * w
            total_abs += abs(w)
    if total_abs <= 0:
        return out
    return out / total_abs


def add_score_percentile(score_df):
    out = score_df.copy()
    out["score_rank_pct"] = np.nan
    group_cols = ["model_name", "score_family", "target_name", "train_tag", "feature_date"]
    for _, idx in out.groupby(group_cols).groups.items():
        s = out.loc[idx, "score"].astype(float)
        out.loc[idx, "score_rank_pct"] = s.rank(pct=True)
    return out


In [ ]:
# =========================
# 4. Train LGB scores and build manual anchor scores
# =========================
SCORE_OUTPUT_COLS = [
    "code", "name", "feature_date", "rebalance_date", "next_date",
    RISK_BREADTH_COL, "future_ret", "target_value", "score",
    "score_family", "target_name", "target_col", "eval_ret_col",
    "horizon_days", "train_tag", "model_name",
]

def append_csv(df, path):
    if df is None or df.empty:
        return
    out = df.copy()
    for col in SCORE_OUTPUT_COLS:
        if col not in out.columns:
            out[col] = np.nan
    out = out.loc[:, SCORE_OUTPUT_COLS]
    write_header = not os.path.exists(path)
    out.to_csv(path, mode="a", header=write_header, index=False)


def unique_existing_cols(cols, df):
    out = []
    for col in cols:
        if col in df.columns and col not in out:
            out.append(col)
    return out


def first_col_as_series(df, col):
    value = df.loc[:, col]
    if isinstance(value, pd.DataFrame):
        return value.iloc[:, 0]
    return value


def save_path_panel_cache(df):
    path_cols = [c for c in PATH_RET_COLS if c in df.columns]
    if len(path_cols) == 0:
        print("path panel cache skipped: no path columns in panel_df")
        return
    if os.path.exists(PATH_PANEL_CSV):
        print("reuse path panel cache:", PATH_PANEL_CSV)
        return
    cols = unique_existing_cols(["code", "feature_date"] + path_cols, df)
    df.to_csv(PATH_PANEL_CSV, columns=cols, index=False)
    print("saved path panel cache:", PATH_PANEL_CSV, "cols", len(cols))


def export_lgb_bundle(model, fill_values, target_name, target_col, train_tag, backend):
    if not EXPORT_MODELS:
        return ""
    if not os.path.exists(MODEL_BUNDLE_DIR):
        os.makedirs(MODEL_BUNDLE_DIR)
    filename = "model_etf_ml_v6_lgb_%s_%s.pkl" % (target_name, train_tag)
    model_path = os.path.join(MODEL_BUNDLE_DIR, filename)
    bundle = {
        "objective": "etf_ml_weekly_lgb_v6",
        "research_version": "etf_ml_v6_anchor_ml_filter_dedup_governance",
        "model": model,
        "backend": backend,
        "target_name": target_name,
        "target_col": target_col,
        "feature_cols": list(FEATURE_COLS),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS),
        "context_feature_cols": list(POOL_FEATURE_COLS),
        "fill_values": dict(fill_values),
        "lookback_days": int(LOOKBACK_DAYS),
        "trend_windows": list(TREND_WINDOWS),
        "trend_weight_end": float(TREND_WEIGHT_END),
        "price_history_count": int(PRICE_HISTORY_COUNT),
        "pool_context_window": int(POOL_CONTEXT_WINDOW),
        "min_listing_days": int(MIN_LISTING_DAYS),
        "min_avg_money_20": float(MIN_AVG_MONEY_20),
        "stock_num": 3,
        "benchmark": BENCHMARK,
        "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
        "use_orthogonal_pool": bool(USE_ORTHOGONAL_POOL),
        "orthogonal_pool_path": ORTHOGONAL_POOL_PATH,
        "train_tag": train_tag,
    }
    with open(model_path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)
    return model_path

for p in [SCORE_CSV, MODEL_MANIFEST_CSV]:
    if os.path.exists(p):
        os.remove(p)

save_path_panel_cache(panel_df)
panel_df = clean_feature_frame(panel_df, include_path=False)
gc.collect()
print("training panel shape after dropping path cols:", panel_df.shape)
manifest_rows = []

for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc="train windows"):
    train_start_ts = pd.Timestamp(train_start)
    train_end_ts = pd.Timestamp(train_end)
    eval_mask_base = panel_df["feature_date"] > train_end_ts
    if not bool(eval_mask_base.any()):
        print("skip eval empty", train_tag)
        continue

    eval_base = panel_df.loc[eval_mask_base].copy()
    eval_base["manual_trend_score"] = calc_manual_trend_score(eval_base)

    manual_done = set()
    for spec in tqdm(TARGET_SPECS, desc=train_tag, leave=False):
        target_name = spec["target_name"]
        target_col = spec["target_col"]
        eval_ret_col = spec["eval_ret_col"]
        next_col = spec["next_col"]
        eval_horizon_days = int(spec["eval_horizon_days"])
        if target_col not in panel_df.columns or eval_ret_col not in panel_df.columns or next_col not in panel_df.columns:
            print("skip missing target columns", target_name)
            continue

        train_mask = (
            (panel_df["feature_date"] >= train_start_ts) &
            (panel_df[next_col] <= train_end_ts) &
            (~panel_df[target_col].isnull())
        )
        eval_mask = eval_mask_base & (~panel_df[eval_ret_col].isnull()) & (~panel_df[target_col].isnull())
        train_df = panel_df.loc[train_mask].copy()
        score_df_h = panel_df.loc[eval_mask].copy()
        if train_df.empty or score_df_h.empty:
            print("skip", train_tag, target_name, "train", train_df.shape, "eval", score_df_h.shape)
            continue

        print("training ML", train_tag, target_name, "samples", train_df.shape[0], "weeks", train_df["feature_date"].nunique())
        model, fill_values, backend = train_lgb_or_fallback(train_df, target_col)
        model_file = export_lgb_bundle(model, fill_values, target_name, target_col, train_tag, backend)
        ml_score = predict_model(model, fill_values, score_df_h)

        base_cols = unique_existing_cols(["code", "name", "feature_date", "rebalance_date", RISK_BREADTH_COL], score_df_h)
        ml_out = score_df_h.loc[:, base_cols].copy()
        ml_out["future_ret"] = first_col_as_series(score_df_h, eval_ret_col)
        ml_out["target_value"] = first_col_as_series(score_df_h, target_col)
        ml_out["next_date"] = first_col_as_series(score_df_h, next_col)
        ml_out["score"] = ml_score
        ml_out["score_family"] = "ml_lgb"
        ml_out["target_name"] = target_name
        ml_out["target_col"] = target_col
        ml_out["eval_ret_col"] = eval_ret_col
        ml_out["horizon_days"] = eval_horizon_days
        ml_out["train_tag"] = train_tag
        ml_out["model_name"] = "ml_lgb_%s_%s" % (target_name, train_tag)
        append_csv(ml_out, SCORE_CSV)

        manifest_rows.append({
            "score_family": "ml_lgb",
            "target_name": target_name,
            "target_col": target_col,
            "eval_ret_col": eval_ret_col,
            "horizon_days": eval_horizon_days,
            "train_tag": train_tag,
            "train_start": train_start,
            "train_end": train_end,
            "train_samples": int(train_df.shape[0]),
            "eval_samples": int(score_df_h.shape[0]),
            "backend": backend,
            "model_file": model_file,
        })

        manual_target = manual_target_for_eval_horizon(eval_horizon_days)
        if manual_target not in manual_done:
            man_eval_ret_col = MANUAL_EVAL_RET_COL
            man_next_col = MANUAL_NEXT_COL
            if man_eval_ret_col not in eval_base.columns or man_next_col not in eval_base.columns:
                print("skip manual anchor, missing", man_eval_ret_col, man_next_col)
            else:
                man_eval = eval_base[(~eval_base[man_eval_ret_col].isnull())].copy()
                man_keep_cols = unique_existing_cols(["code", "name", "feature_date", "rebalance_date", RISK_BREADTH_COL], man_eval)
                man_out = man_eval.loc[:, man_keep_cols].copy()
                man_out["future_ret"] = first_col_as_series(man_eval, man_eval_ret_col)
                man_out["target_value"] = man_out["future_ret"]
                man_out["next_date"] = first_col_as_series(man_eval, man_next_col)
                man_out["score"] = calc_manual_trend_score(man_eval)
                man_out["score_family"] = "manual_trend_r2"
                man_out["target_name"] = manual_target
                man_out["target_col"] = "manual_score"
                man_out["eval_ret_col"] = man_eval_ret_col
                man_out["horizon_days"] = int(MANUAL_HORIZON_DAYS)
                man_out["train_tag"] = train_tag
                man_out["model_name"] = "manual_trend_r2_%s" % train_tag
                append_csv(man_out, SCORE_CSV)
                manifest_rows.append({
                    "score_family": "manual_trend_r2",
                    "target_name": manual_target,
                    "target_col": "manual_score",
                    "eval_ret_col": man_eval_ret_col,
                    "horizon_days": int(MANUAL_HORIZON_DAYS),
                    "train_tag": train_tag,
                    "train_start": train_start,
                    "train_end": train_end,
                    "train_samples": 0,
                    "eval_samples": int(man_out.shape[0]),
                    "backend": "manual_formula",
                    "model_file": "",
                })
            manual_done.add(manual_target)

        del train_df, score_df_h, ml_out, model
        gc.collect()
    del eval_base
    gc.collect()

if os.path.exists(SCORE_CSV):
    score_df = pd.read_csv(SCORE_CSV)
    for _dt_col in ["feature_date", "rebalance_date", "next_date"]:
        if _dt_col in score_df.columns:
            score_df[_dt_col] = pd.to_datetime(score_df[_dt_col])
else:
    score_df = pd.DataFrame()
if not score_df.empty:
    score_df = add_score_percentile(score_df)
    score_df.to_csv(SCORE_CSV, index=False)
model_manifest_df = pd.DataFrame(manifest_rows)
model_manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)
print("score shape:", score_df.shape)
print("manifest shape:", model_manifest_df.shape)
display(model_manifest_df.head(30))





In [ ]:
# =========================
# 5. Portfolio proxy: raw top, dynamic de-dup, and manual+ML filter
# =========================
def summarize_returns(ret_series):
    r = pd.Series(ret_series).dropna().astype(float)
    if len(r) == 0:
        return {"periods": 0}
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    mean_ret = float(r.mean())
    std_ret = float(r.std())
    sharpe = np.nan if std_ret <= 0 or pd.isnull(std_ret) else mean_ret / std_ret
    return {
        "periods": int(len(r)),
        "cum_ret": float(nav.iloc[-1] - 1.0),
        "mean_period_ret": mean_ret,
        "win_rate": float((r > 0).mean()),
        "period_sharpe": float(sharpe) if not pd.isnull(sharpe) else np.nan,
        "max_drawdown": float(dd.min()),
    }


def get_path_matrix(df):
    cols = [c for c in PATH_RET_COLS if c in df.columns]
    if len(cols) == 0 or df.empty:
        return pd.DataFrame(index=df.index)
    mat = df.loc[:, cols].replace([np.inf, -np.inf], np.nan).copy()
    return mat


def pair_corr_from_paths(row_a, row_b):
    a = pd.Series(row_a).astype(float)
    b = pd.Series(row_b).astype(float)
    mask = (~a.isnull()) & (~b.isnull())
    if int(mask.sum()) < DEDUP_MIN_PATH_OBS:
        return np.nan
    return float(a[mask].corr(b[mask]))


def avg_pair_corr(top):
    if top is None or top.empty or len(top) <= 1:
        return np.nan
    mat = get_path_matrix(top)
    vals = []
    idxs = list(mat.index)
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            vals.append(pair_corr_from_paths(mat.loc[idxs[i]], mat.loc[idxs[j]]))
    vals = [v for v in vals if not pd.isnull(v)]
    return float(np.mean(vals)) if vals else np.nan


PATH_PANEL_DF = None

def load_path_panel_cache():
    global PATH_PANEL_DF
    if PATH_PANEL_DF is not None:
        return PATH_PANEL_DF
    if not os.path.exists(PATH_PANEL_CSV):
        PATH_PANEL_DF = pd.DataFrame()
        print("path panel cache missing, dedup modes will fallback to raw top:", PATH_PANEL_CSV)
        return PATH_PANEL_DF
    usecols = ["code", "feature_date"] + PATH_RET_COLS
    try:
        PATH_PANEL_DF = pd.read_csv(PATH_PANEL_CSV, usecols=lambda c: c in usecols)
    except TypeError:
        PATH_PANEL_DF = pd.read_csv(PATH_PANEL_CSV)
        keep = [c for c in usecols if c in PATH_PANEL_DF.columns]
        PATH_PANEL_DF = PATH_PANEL_DF.loc[:, keep].copy()
    if "feature_date" in PATH_PANEL_DF.columns:
        PATH_PANEL_DF["feature_date"] = pd.to_datetime(PATH_PANEL_DF["feature_date"])
    print("loaded path panel cache:", PATH_PANEL_DF.shape)
    return PATH_PANEL_DF

def attach_path_columns_for_week(gdf):
    if gdf.empty:
        return gdf
    path_panel = load_path_panel_cache()
    path_cols = [c for c in PATH_RET_COLS if c in path_panel.columns]
    if len(path_cols) == 0 or path_panel.empty:
        return gdf
    feature_date = pd.Timestamp(gdf["feature_date"].iloc[0])
    path_df = path_panel.loc[path_panel["feature_date"] == feature_date, ["code"] + path_cols].copy()
    if path_df.empty:
        return gdf
    out = gdf.merge(path_df, on="code", how="left")
    del path_df
    return out


def select_raw_top(sorted_df, select_mode):
    n = 1 if select_mode == "top1" else 3
    return sorted_df.head(min(n, len(sorted_df))).copy()


def select_corr_dedup(sorted_df, candidate_n, target_n, corr_threshold):
    cand = sorted_df.head(min(candidate_n, len(sorted_df))).copy()
    selected_idx = []
    path_mat = get_path_matrix(cand)
    for idx, row in cand.iterrows():
        if len(selected_idx) >= target_n:
            break
        ok = True
        for sidx in selected_idx:
            corr = pair_corr_from_paths(path_mat.loc[idx], path_mat.loc[sidx]) if idx in path_mat.index and sidx in path_mat.index else np.nan
            if (not pd.isnull(corr)) and corr >= corr_threshold:
                ok = False
                break
        if ok:
            selected_idx.append(idx)
    if len(selected_idx) < target_n:
        for idx in cand.index:
            if idx not in selected_idx:
                selected_idx.append(idx)
                if len(selected_idx) >= target_n:
                    break
    return cand.loc[selected_idx].copy()


def select_kmeans_top(sorted_df, candidate_n, target_n):
    cand = sorted_df.head(min(candidate_n, len(sorted_df))).copy()
    if len(cand) <= target_n:
        return cand.copy()
    mat = get_path_matrix(cand)
    mat = mat.dropna(axis=1, how="all")
    if mat.empty or mat.shape[1] < DEDUP_MIN_PATH_OBS:
        return select_corr_dedup(sorted_df, candidate_n, target_n, DEDUP_CORR_THRESHOLD)
    mat = mat.fillna(0.0)
    try:
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=target_n, random_state=RANDOM_SEED)
        labels = km.fit_predict(mat.values)
    except Exception as err:
        print("kmeans failed, fallback corr dedup:", err)
        return select_corr_dedup(sorted_df, candidate_n, target_n, DEDUP_CORR_THRESHOLD)
    cand["cluster_id"] = labels
    picked = []
    for _, gdf in cand.groupby("cluster_id"):
        picked.append(gdf.index[0])
    if len(picked) < target_n:
        for idx in cand.index:
            if idx not in picked:
                picked.append(idx)
                if len(picked) >= target_n:
                    break
    return cand.loc[picked[:target_n]].copy()


def select_by_mode(sorted_df, select_mode):
    if select_mode in ["top1", "top3"]:
        return select_raw_top(sorted_df, select_mode)
    if select_mode == "corr_dedup_top3":
        return select_corr_dedup(sorted_df, DEDUP_CANDIDATE_N, DEDUP_TARGET_N, DEDUP_CORR_THRESHOLD)
    if select_mode == "kmeans_top3":
        return select_kmeans_top(sorted_df, DEDUP_CANDIDATE_N, DEDUP_TARGET_N)
    return select_raw_top(sorted_df, "top3")


def build_weekly_portfolio(score_data):
    rows = []
    rng = np.random.RandomState(RANDOM_SEED)
    group_cols = ["model_name", "score_family", "target_name", "horizon_days", "train_tag", "feature_date"]
    for keys, gdf in tqdm(score_data.groupby(group_cols), desc="portfolio proxy"):
        model_name, score_family, target_name, h, train_tag, feature_date = keys
        gdf = gdf.sort_values("score", ascending=False).copy()
        gdf = attach_path_columns_for_week(gdf)
        if gdf.empty:
            continue
        median_ret = float(gdf["future_ret"].median())
        for select_mode in SELECT_MODES:
            top = select_by_mode(gdf, select_mode)
            raw_ret = float(top["future_ret"].mean()) if not top.empty else np.nan
            random_n = len(top)
            random_ret = float(gdf.sample(random_n, random_state=int(rng.randint(0, 1000000)))["future_ret"].mean()) if random_n > 0 else np.nan
            rows.append({
                "strategy_family": "direct_select",
                "model_name": model_name,
                "score_family": score_family,
                "target_name": target_name,
                "horizon_days": int(h),
                "train_tag": train_tag,
                "feature_date": feature_date,
                "next_date": top["next_date"].iloc[0] if not top.empty else pd.NaT,
                "select_mode": select_mode,
                "risk_mode": "none",
                "risk_off": False,
                "position_weight": 1.0,
                "pool_breadth": float(gdf[RISK_BREADTH_COL].dropna().iloc[0]) if RISK_BREADTH_COL in gdf.columns and len(gdf[RISK_BREADTH_COL].dropna()) else np.nan,
                "target_count": int(len(top)),
                "targets": ",".join(top["code"].astype(str).tolist()),
                "target_names": ",".join(top.get("name", pd.Series([""] * len(top))).astype(str).tolist()),
                "avg_pair_corr": avg_pair_corr(top),
                "raw_portfolio_ret": raw_ret,
                "portfolio_ret": raw_ret,
                "universe_median_ret": median_ret,
                "excess_vs_median": raw_ret - median_ret if not pd.isnull(raw_ret) else np.nan,
                "random_ret": random_ret,
                "excess_vs_random": raw_ret - random_ret if not pd.isnull(raw_ret) and not pd.isnull(random_ret) else np.nan,
                "ml_rank_pct": np.nan,
            })
    return pd.DataFrame(rows)


def build_manual_ml_filter_portfolio(score_data):
    rows = []
    if score_data.empty:
        return pd.DataFrame()
    ml_data = score_data[score_data["score_family"] == "ml_lgb"].copy()
    man_data = score_data[score_data["score_family"] == "manual_trend_r2"].copy()
    group_cols = ["model_name", "target_name", "horizon_days", "train_tag", "feature_date"]
    for keys, ml_gdf in tqdm(ml_data.groupby(group_cols), desc="manual+ml filter"):
        ml_model, ml_target, h, train_tag, feature_date = keys
        manual_target = manual_target_for_eval_horizon(int(h))
        manual_model = "manual_trend_r2_%s" % train_tag
        man_gdf = man_data[(man_data["model_name"] == manual_model) & (man_data["feature_date"] == feature_date)].copy()
        if man_gdf.empty:
            continue
        manual_top = man_gdf.sort_values("score", ascending=False).head(1).copy()
        if manual_top.empty:
            continue
        code = str(manual_top["code"].iloc[0])
        ml_row = ml_gdf[ml_gdf["code"].astype(str) == code].copy()
        if ml_row.empty:
            continue
        ml_rank_pct = float(ml_row["score_rank_pct"].iloc[0]) if "score_rank_pct" in ml_row.columns else np.nan
        raw_ret = float(manual_top["future_ret"].iloc[0])
        median_ret = float(man_gdf["future_ret"].median())
        name = manual_top.get("name", pd.Series([""])).iloc[0]
        next_date = manual_top["next_date"].iloc[0]
        pool_breadth = float(manual_top[RISK_BREADTH_COL].iloc[0]) if RISK_BREADTH_COL in manual_top.columns else np.nan
        for mode in ["filter", "scale"]:
            if mode == "filter":
                weight = 1.0 if (not pd.isnull(ml_rank_pct) and ml_rank_pct >= FILTER_BUY_RANK_PCT) else 0.0
            else:
                if pd.isnull(ml_rank_pct) or ml_rank_pct < SCALE_LOW_RANK_PCT:
                    weight = 0.0
                elif ml_rank_pct < SCALE_HIGH_RANK_PCT:
                    weight = 0.5
                else:
                    weight = 1.0
            realized_ret = raw_ret * weight
            rows.append({
                "strategy_family": "manual_top1_ml_%s" % mode,
                "model_name": "manual_top1_%s_%s" % (mode, ml_model),
                "score_family": "manual_ml_%s" % mode,
                "target_name": ml_target,
                "horizon_days": int(h),
                "train_tag": train_tag,
                "feature_date": feature_date,
                "next_date": next_date,
                "select_mode": "manual_top1_ml_%s" % mode,
                "risk_mode": "none",
                "risk_off": bool(weight <= 0),
                "position_weight": weight,
                "pool_breadth": pool_breadth,
                "target_count": 1 if weight > 0 else 0,
                "targets": code if weight > 0 else "",
                "target_names": str(name) if weight > 0 else "",
                "avg_pair_corr": np.nan,
                "raw_portfolio_ret": raw_ret,
                "portfolio_ret": realized_ret,
                "universe_median_ret": median_ret,
                "excess_vs_median": realized_ret - median_ret,
                "random_ret": np.nan,
                "excess_vs_random": np.nan,
                "ml_rank_pct": ml_rank_pct,
            })
    return pd.DataFrame(rows)

weekly_direct_df = build_weekly_portfolio(score_df) if not score_df.empty else pd.DataFrame()
weekly_filter_df = build_manual_ml_filter_portfolio(score_df) if not score_df.empty else pd.DataFrame()
weekly_df = pd.concat([weekly_direct_df, weekly_filter_df], ignore_index=True) if (not weekly_filter_df.empty) else weekly_direct_df
assert_no_excluded_names(weekly_df, "target_names", "weekly_df")
weekly_df.to_csv(WEEKLY_CSV, index=False)
print("weekly proxy shape:", weekly_df.shape)
display(weekly_df.head())



In [ ]:
# =========================
# 6. Summary, de-dup report, filter report, and governance
# =========================
def build_summary(weekly):
    rows = []
    if weekly.empty:
        return pd.DataFrame()
    group_cols = ["strategy_family", "score_family", "target_name", "horizon_days", "train_tag", "select_mode", "risk_mode", "model_name"]
    for keys, gdf in weekly.groupby(group_cols):
        strategy_family, score_family, target_name, h, train_tag, select_mode, risk_mode, model_name = keys
        ret_stats = summarize_returns(gdf["portfolio_ret"])
        med_stats = summarize_returns(gdf["excess_vs_median"])
        rnd_stats = summarize_returns(gdf["excess_vs_random"])
        row = {
            "strategy_family": strategy_family,
            "score_family": score_family,
            "target_name": target_name,
            "horizon_days": int(h),
            "train_tag": train_tag,
            "select_mode": select_mode,
            "risk_mode": risk_mode,
            "model_name": model_name,
            "risk_off_rate": float(gdf["risk_off"].mean()) if "risk_off" in gdf.columns else 0.0,
            "avg_position_weight": float(gdf["position_weight"].mean()) if "position_weight" in gdf.columns else 1.0,
            "avg_pool_breadth": float(gdf["pool_breadth"].mean()),
            "avg_pair_corr": float(gdf["avg_pair_corr"].mean()) if "avg_pair_corr" in gdf.columns else np.nan,
            "avg_ml_rank_pct": float(gdf["ml_rank_pct"].mean()) if "ml_rank_pct" in gdf.columns else np.nan,
        }
        row.update({"ret_" + k: v for k, v in ret_stats.items()})
        row.update({"excess_median_" + k: v for k, v in med_stats.items()})
        row.update({"excess_random_" + k: v for k, v in rnd_stats.items()})
        rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["ret_cum_ret", "ret_max_drawdown"], ascending=[False, False])
    return out


def build_dedup_report(summary):
    if summary.empty:
        return pd.DataFrame()
    base = summary[(summary["strategy_family"] == "direct_select") & (summary["select_mode"] == "top3")].copy()
    rows = []
    key_cols = ["model_name", "score_family", "target_name", "horizon_days", "train_tag"]
    for _, b in base.iterrows():
        for mode in ["corr_dedup_top3", "kmeans_top3"]:
            cand = summary[(summary["strategy_family"] == "direct_select") & (summary["select_mode"] == mode)].copy()
            for col in key_cols:
                cand = cand[cand[col] == b[col]]
            if cand.empty:
                continue
            r = cand.iloc[0]
            rows.append({
                "model_name": b["model_name"],
                "score_family": b["score_family"],
                "target_name": b["target_name"],
                "horizon_days": b["horizon_days"],
                "train_tag": b["train_tag"],
                "dedup_mode": mode,
                "base_cum_ret": b.get("ret_cum_ret", np.nan),
                "dedup_cum_ret": r.get("ret_cum_ret", np.nan),
                "cum_ret_delta": r.get("ret_cum_ret", np.nan) - b.get("ret_cum_ret", np.nan),
                "base_max_drawdown": b.get("ret_max_drawdown", np.nan),
                "dedup_max_drawdown": r.get("ret_max_drawdown", np.nan),
                "max_dd_improvement": r.get("ret_max_drawdown", np.nan) - b.get("ret_max_drawdown", np.nan),
                "base_avg_pair_corr": b.get("avg_pair_corr", np.nan),
                "dedup_avg_pair_corr": r.get("avg_pair_corr", np.nan),
                "avg_pair_corr_delta": r.get("avg_pair_corr", np.nan) - b.get("avg_pair_corr", np.nan),
            })
    return pd.DataFrame(rows).sort_values("cum_ret_delta", ascending=False) if rows else pd.DataFrame()


def build_filter_report(summary):
    if summary.empty:
        return pd.DataFrame()
    rows = []
    filters = summary[summary["strategy_family"].isin(["manual_top1_ml_filter", "manual_top1_ml_scale"])].copy()
    for _, f in filters.iterrows():
        manual_target = manual_target_for_eval_horizon(int(f["horizon_days"]))
        manual_model = "manual_trend_r2_%s" % f["train_tag"]
        base = summary[(summary["model_name"] == manual_model) & (summary["select_mode"] == "top1")].copy()
        if base.empty:
            continue
        b = base.iloc[0]
        rows.append({
            "filter_model_name": f["model_name"],
            "filter_target_name": f["target_name"],
            "filter_mode": f["strategy_family"],
            "train_tag": f["train_tag"],
            "horizon_days": f["horizon_days"],
            "base_manual_model": manual_model,
            "base_cum_ret": b.get("ret_cum_ret", np.nan),
            "filter_cum_ret": f.get("ret_cum_ret", np.nan),
            "cum_ret_delta": f.get("ret_cum_ret", np.nan) - b.get("ret_cum_ret", np.nan),
            "base_max_drawdown": b.get("ret_max_drawdown", np.nan),
            "filter_max_drawdown": f.get("ret_max_drawdown", np.nan),
            "max_dd_improvement": f.get("ret_max_drawdown", np.nan) - b.get("ret_max_drawdown", np.nan),
            "avg_position_weight": f.get("avg_position_weight", np.nan),
            "risk_off_rate": f.get("risk_off_rate", np.nan),
        })
    return pd.DataFrame(rows).sort_values("cum_ret_delta", ascending=False) if rows else pd.DataFrame()


def calc_window_stats(rows):
    if rows is None or len(rows) == 0:
        return None
    r = pd.Series(rows).dropna().astype(float)
    if len(r) == 0:
        return None
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return {
        "cum_ret": float(nav.iloc[-1] - 1.0),
        "max_drawdown": float(dd.min()),
        "score": float(nav.iloc[-1] - 1.0 + 0.5 * dd.min()),
        "obs": int(len(r)),
    }


def build_governance_report(weekly):
    if weekly.empty:
        return pd.DataFrame()
    src = weekly[weekly["risk_mode"] == "none"].copy()
    src["strategy_id"] = src["model_name"].astype(str) + "|" + src["select_mode"].astype(str)
    rows = []
    for train_tag, tdf in src.groupby("train_tag"):
        dates = sorted(tdf["feature_date"].dropna().unique())
        for lookback in GOVERNANCE_LOOKBACK_LIST:
            for dt in dates:
                hist = tdf[tdf["feature_date"] < dt].copy()
                cur = tdf[tdf["feature_date"] == dt].copy()
                if hist.empty or cur.empty:
                    continue
                candidates = []
                for sid, hdf in hist.groupby("strategy_id"):
                    hdf = hdf.sort_values("feature_date").tail(lookback)
                    stats = calc_window_stats(hdf["portfolio_ret"].tolist())
                    if stats is None or stats["obs"] < GOVERNANCE_MIN_OBS:
                        continue
                    if stats["max_drawdown"] < GOVERNANCE_MAX_DD_FLOOR:
                        continue
                    candidates.append((stats["score"], sid, stats))
                if len(candidates) == 0:
                    rows.append({
                        "train_tag": train_tag,
                        "lookback_weeks": lookback,
                        "feature_date": dt,
                        "selected_strategy_id": "cash",
                        "selected_score": np.nan,
                        "selected_hist_cum_ret": np.nan,
                        "selected_hist_max_drawdown": np.nan,
                        "portfolio_ret": CASH_RETURN,
                        "risk_off": True,
                    })
                    continue
                candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
                best_score, best_sid, best_stats = candidates[0]
                cur_best = cur[cur["strategy_id"] == best_sid].copy()
                realized = float(cur_best["portfolio_ret"].iloc[0]) if not cur_best.empty else CASH_RETURN
                rows.append({
                    "train_tag": train_tag,
                    "lookback_weeks": lookback,
                    "feature_date": dt,
                    "selected_strategy_id": best_sid,
                    "selected_score": best_score,
                    "selected_hist_cum_ret": best_stats["cum_ret"],
                    "selected_hist_max_drawdown": best_stats["max_drawdown"],
                    "portfolio_ret": realized,
                    "risk_off": False if not cur_best.empty else True,
                })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    summary_rows = []
    for keys, gdf in out.groupby(["train_tag", "lookback_weeks"]):
        train_tag, lookback = keys
        s = summarize_returns(gdf.sort_values("feature_date")["portfolio_ret"])
        row = {"train_tag": train_tag, "lookback_weeks": lookback, "risk_off_rate": float(gdf["risk_off"].mean())}
        row.update({"gov_" + k: v for k, v in s.items()})
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    out.to_csv(GOVERNANCE_REPORT_CSV.replace(".csv", "_detail.csv"), index=False)
    return summary.sort_values("gov_cum_ret", ascending=False)

summary_df = build_summary(weekly_df)
dedup_report_df = build_dedup_report(summary_df)
filter_report_df = build_filter_report(summary_df)
governance_report_df = build_governance_report(weekly_df)
strategy_compare_df = summary_df.copy()

summary_df.to_csv(SUMMARY_CSV, index=False)
dedup_report_df.to_csv(DEDUP_REPORT_CSV, index=False)
filter_report_df.to_csv(FILTER_REPORT_CSV, index=False)
governance_report_df.to_csv(GOVERNANCE_REPORT_CSV, index=False)
strategy_compare_df.to_csv(STRATEGY_COMPARE_CSV, index=False)

print("summary top:")
display(summary_df.head(30))
print("dedup report top:")
display(dedup_report_df.head(20))
print("filter report top:")
display(filter_report_df.head(20))
print("governance report:")
display(governance_report_df)


In [ ]:
# =========================
# 7. Latest targets and output manifest
# =========================
latest_rows = []
if not score_df.empty:
    for (model_name, score_family, target_name), gdf in score_df.groupby(["model_name", "score_family", "target_name"]):
        latest_date = gdf["feature_date"].max()
        latest_sorted = gdf[gdf["feature_date"] == latest_date].sort_values("score", ascending=False).copy()
        for mode in SELECT_MODES:
            top = select_by_mode(latest_sorted, mode)
            for rank_idx, (_, row) in enumerate(top.iterrows(), 1):
                latest_rows.append({
                    "model_name": row["model_name"],
                    "score_family": row["score_family"],
                    "target_name": row["target_name"],
                    "horizon_days": row["horizon_days"],
                    "train_tag": row["train_tag"],
                    "feature_date": latest_date,
                    "select_mode": mode,
                    "rank": rank_idx,
                    "code": row["code"],
                    "name": row.get("name", ""),
                    "score": row["score"],
                    "score_rank_pct": row.get("score_rank_pct", np.nan),
                    "future_ret": row.get("future_ret", np.nan),
                    "pool_breadth": row.get(RISK_BREADTH_COL, np.nan),
                })
latest_targets_df = pd.DataFrame(latest_rows)
assert_no_excluded_names(latest_targets_df, "name", "latest_targets_df")
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)
display(latest_targets_df.head(80))

print("outputs saved:")
for p in [PANEL_CSV, SCORE_CSV, WEEKLY_CSV, SUMMARY_CSV, STRATEGY_COMPARE_CSV, DEDUP_REPORT_CSV, FILTER_REPORT_CSV, GOVERNANCE_REPORT_CSV, LATEST_TARGETS_CSV, MODEL_MANIFEST_CSV]:
    print(" -", p)

print("V6 conclusion template:")
print("1) 先看 summary/strategy_compare 的 top direct_select。")
print("2) 再看 dedup_report 判断 corr/kmeans 是否减少相关性且不牺牲收益。")
print("3) 再看 filter_report 判断 manual top1 + ML filter/scale 是否改善回撤或收益回撤比。")
print("4) 最后看 governance_report 判断策略切换是否有正向意义。")

## Self Review

- 不再使用 ETF 行业/主题硬去重；`top3` 是原始分数前三名。
- ML 训练样本用 `next_date_h <= train_end`，OOS 评估用 `feature_date > train_end`，避免训练截止日标签穿越。
- 手动趋势模型不使用未来收益，只使用 feature date 及以前构造的趋势/R²/动量/波动 rank。
- 风险层只使用同周 ETF 池内趋势广度，低于阈值时按现金收益处理；这是组合风险层，不是模型训练特征泄漏。
- 离线 proxy 是 close-to-close，用来筛方向；进入实盘/回测前，仍需用 JoinQuant 回测文件验证 09:35 成交、佣金、滑点、持仓路径。
- 长循环使用 `tqdm`，如果聚宽环境缺少 `tqdm` 会 fallback 为普通循环。